In [ ]:
# ============================================================
# CELL 1 — Mount Drive & Install dependencies
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# openai/CLIP official package
!pip install -q git+https://github.com/openai/CLIP.git

In [ ]:
# ============================================================
# CELL 2 — Configuration
# ============================================================
import os

BASE_DATASET = '/content/drive/MyDrive/Video_Sum_Dataset'

# All three dataset folders
FRAMES_ROOTS = {
    'Activity':  os.path.join(BASE_DATASET, 'Activity/frames'),
    'Animals':   os.path.join(BASE_DATASET, 'Animals/frames'),
    'Incidents': os.path.join(BASE_DATASET, 'Incidents/frames'),
}

# Separate folder from the ViT-L/14 (768-dim) embeddings
EMBEDDINGS_ROOT = os.path.join(BASE_DATASET, 'embeddings_clip_rn50x64')
os.makedirs(EMBEDDINGS_ROOT, exist_ok=True)

BATCH_SIZE = 32   # RN50x64 is heavier than ViT-L/14 — use 32 to avoid OOM on T4
DEVICE = 'cuda'   # Colab T4/A100 — will fall back to cpu automatically

In [ ]:
# ============================================================
# CELL 3 — Load CLIP RN50x64 (dim=1024)
# ============================================================
import torch
import clip

device = torch.device(DEVICE if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# RN50x64 is the only OpenAI CLIP variant that outputs true 1024-dim embeddings.
# It uses a widened ResNet backbone (not a ViT).
# clip.load() downloads weights (~3.5 GB) once and returns:
#   model      — the full CLIP model (we only use the visual encoder)
#   preprocess — the official preprocessing transform (448x448 input for RN50x64)
model, preprocess = clip.load('RN50x64', device=device)
model.eval()

# Quick sanity check on the output dimension
# RN50x64 expects 448x448 input — use that for the dummy
with torch.no_grad():
    dummy = torch.zeros(1, 3, 448, 448).to(device)
    dim = model.encode_image(dummy).shape[-1]

print(f"\u2713 CLIP RN50x64 loaded")
print(f"  Output embedding dim : {dim}")
print(f"  Parameters           : {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

In [ ]:
# ============================================================
# CELL 4 — Define preprocessing & embedding extraction
# ============================================================
import torch
import numpy as np
from PIL import Image
import glob

# 'preprocess' was already returned by clip.load() in Cell 3.
# For RN50x64 it handles: Resize(448) → CenterCrop(448) → ToTensor → Normalize
# using the exact mean/std CLIP was trained with — no manual config needed.

def load_image_batch(paths):
    """Load a list of image paths into a batched tensor using CLIP's preprocess."""
    tensors = []
    valid_paths = []
    for p in paths:
        try:
            img = Image.open(p).convert('RGB')
            tensors.append(preprocess(img))
            valid_paths.append(p)
        except Exception as e:
            print(f"  \u26a0 Skipping {os.path.basename(p)}: {e}")
    if not tensors:
        return None, []
    return torch.stack(tensors), valid_paths


@torch.no_grad()
def extract_embeddings_for_video(frame_dir, batch_size=64):
    """
    Given a folder of frames (frame_0001.jpg, ...),
    returns:
        embeddings : np.ndarray of shape (N, 1024)  — L2-normalised CLIP RN50x64 features
        frame_names: list of N filenames (sorted)
    """
    frame_paths = sorted(glob.glob(os.path.join(frame_dir, '*.jpg')))
    if not frame_paths:
        return None, []

    all_embeddings = []
    all_names = []

    for i in range(0, len(frame_paths), batch_size):
        batch_paths = frame_paths[i:i + batch_size]
        batch_tensor, valid_paths = load_image_batch(batch_paths)
        if batch_tensor is None:
            continue

        batch_tensor = batch_tensor.to(device)

        # encode_image returns the visual CLS embedding: (B, 1024)  float32
        # We L2-normalise so cosine similarity = dot product downstream.
        features = model.encode_image(batch_tensor)          # (B, 1024)
        features = features / features.norm(dim=-1, keepdim=True)  # L2-normalise
        all_embeddings.append(features.cpu().float().numpy())
        all_names.extend([os.path.basename(p) for p in valid_paths])

    if not all_embeddings:
        return None, []

    return np.vstack(all_embeddings), all_names  # (N, 1024)

In [ ]:
# ============================================================
# CELL 5 — Main loop: iterate all folders, save .npz per video
# ============================================================
import time

total_videos = 0
total_frames = 0
skipped = 0

for category, frames_root in FRAMES_ROOTS.items():
    if not os.path.exists(frames_root):
        print(f"\n\u26a0 Frames folder not found, skipping: {frames_root}")
        continue

    print(f"\n{'='*60}")
    print(f"  Category: {category}")
    print(f"  Frames root: {frames_root}")
    print(f"{'='*60}")

    # Each sub-folder = one video
    video_dirs = sorted([
        d for d in os.listdir(frames_root)
        if os.path.isdir(os.path.join(frames_root, d))
    ])

    print(f"  Found {len(video_dirs)} video folders\n")

    # Mirror folder structure: embeddings_clip/<Category>/<video_name>.npz
    category_emb_dir = os.path.join(EMBEDDINGS_ROOT, category)
    os.makedirs(category_emb_dir, exist_ok=True)

    for video_name in video_dirs:
        out_path = os.path.join(category_emb_dir, f"{video_name}.npz")

        # Skip if already computed
        if os.path.exists(out_path):
            existing = np.load(out_path)
            skipped += 1
            print(f"  [SKIP] {video_name} ({existing['embeddings'].shape[0]} frames already embedded)")
            continue

        frame_dir = os.path.join(frames_root, video_name)
        t0 = time.time()

        embeddings, frame_names = extract_embeddings_for_video(frame_dir, BATCH_SIZE)

        if embeddings is None:
            print(f"  [SKIP] {video_name} \u2014 no frames found")
            continue

        # Save as compressed numpy archive
        # Load with: data = np.load('x.npz'); embs = data['embeddings']
        np.savez_compressed(
            out_path,
            embeddings=embeddings,       # shape (N, 1024)  float32  L2-normalised
            frame_names=frame_names,     # list of N filenames
            category=category,
            video_name=video_name,
        )

        elapsed = time.time() - t0
        total_videos += 1
        total_frames += len(frame_names)
        print(f"  \u2713 {video_name}: {len(frame_names)} frames \u2192 (N,1024) | {elapsed:.1f}s")

print(f"\n{'='*60}")
print(f"DONE")
print(f"  Videos processed     : {total_videos}")
print(f"  Videos skipped       : {skipped} (already existed)")
print(f"  Total frames embedded: {total_frames:,}")
print(f"  Embeddings saved to  : {EMBEDDINGS_ROOT}")

In [ ]:
# ============================================================
# CELL 6 — Sanity check: load and inspect one embedding file
# ============================================================
import numpy as np
import os

category_emb_dir = os.path.join(EMBEDDINGS_ROOT, 'Incidents')
sample_file = sorted(os.listdir(category_emb_dir))[0]

data = np.load(os.path.join(category_emb_dir, sample_file))

print(f"File       : {sample_file}")
print(f"Embeddings : {data['embeddings'].shape}  (frames x 1024)")
print(f"dtype      : {data['embeddings'].dtype}")
print(f"Frames     : {list(data['frame_names'])[:5]} ...")
print(f"Category   : {data['category']}")

# L2 norm should be ~1.0 for every row (sanity check on normalisation)
norms = np.linalg.norm(data['embeddings'], axis=1)
print(f"\nL2 norms   : min={norms.min():.4f}  max={norms.max():.4f}  mean={norms.mean():.4f}  (should all be ~1.0)")

print(f"\nFirst embedding vector (first 8 dims):")
print(data['embeddings'][0, :8])